# 中证800 V48 Factor Audit 实验

目的：把问题从“继续调模型”切换为“先确认哪些因子/因子组有稳定信息增量”。

本 notebook 不导出实盘模型，不做新策略增强，先做因子审查：

1. 单因子 RankIC / ICIR / 年度稳定性 / 覆盖率
2. 单因子按训练期 IC 方向构造 top10/top30 月度 alpha 组合
3. 因子组 rank-composite 审查
4. LGB group-only 与 drop-group 诊断，判断某组是否提供模型增量

固定规则：

- 数据：固定读取 `train_csi800_factor_v40_data_enhancement.csv`
- label：`alpha_1m`
- 不调用聚宽 API
- 不使用 early stopping
- 不做 sample weight
- 诊断模型只用于因子审查，不作为回测 pkl


In [ ]:
import os
import gc
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"
OUT_DIR = "csi800_ml_v48_factor_audit_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_COL = "alpha_1m"
TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
DIAG_VALID_FRAC = 0.20
DIAG_VALID_MIN_MONTHS = 6
MODEL_FIXED_ITER = 94
SEED = 42
TOP_LIST = [10, 30]

print("DATA_PATH =", DATA_PATH)
print("OUT_DIR =", OUT_DIR)
print("target =", TARGET_COL)


In [ ]:
FACTOR_GROUPS = {
    "style_processed": [
        "book_to_price_ratio", "earnings_yield", "growth", "liquidity", "momentum", "beta",
    ],
    "value_cashflow": [
        "cash_flow_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "sales_to_price_ratio",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
        "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit",
        "operating_profit_per_share", "net_operate_cash_flow_per_share", "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "ACCA", "growth", "net_working_capital", "MLEV", "debt_to_equity_ratio",
        "debt_to_tangible_equity_ratio", "super_quick_ratio",
    ],
    "momentum_volume": [
        "Rank1M", "sharpe_ratio_60", "VOL10", "DAVOL10", "VMACD", "VOSC",
        "px_close_to_ma60", "ts_Rank1M_rank_chg_1m",
    ],
    "risk_distribution": [
        "Variance20", "beta", "Skewness20", "Kurtosis20", "px_drawdown_60", "liq_paused_count_20",
    ],
    "temporal_liquidity": [
        "liq_money_ratio_20_60", "liq_paused_count_20", "ts_cash_flow_to_price_ratio_rank_mean_3m",
    ],
}

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

ALL_FACTOR_COLS = []
for group_name in FACTOR_GROUPS:
    for col in FACTOR_GROUPS[group_name]:
        if col not in ALL_FACTOR_COLS:
            ALL_FACTOR_COLS.append(col)

print("factor groups:", len(FACTOR_GROUPS))
for k in sorted(FACTOR_GROUPS):
    print(k, len(FACTOR_GROUPS[k]), FACTOR_GROUPS[k])
print("unique factors:", len(ALL_FACTOR_COLS))


In [ ]:
def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def safe_ic_stats(values):
    s = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"mean": np.nan, "std": np.nan, "ir": np.nan, "hit_rate": np.nan, "months": 0}
    std = s.std()
    return {
        "mean": float(s.mean()),
        "std": float(std) if not pd.isnull(std) else np.nan,
        "ir": float(s.mean() / std) if (not pd.isnull(std) and std > 0) else np.nan,
        "hit_rate": float((s > 0).mean()),
        "months": int(len(s)),
    }


def split_diag_valid(df):
    months = sorted(pd.to_datetime(df["rebalance_date"].dropna().unique()))
    n_valid = max(DIAG_VALID_MIN_MONTHS, int(round(len(months) * DIAG_VALID_FRAC)))
    valid_months = set(months[-min(n_valid, max(1, len(months) - 1)):])
    fit = df[~df["rebalance_date"].isin(valid_months)].copy()
    valid = df[df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        fit, valid = df.copy(), df.copy()
    return fit, valid


def prepare_xy(df, feature_cols, fill_values=None):
    d = df.dropna(subset=[TARGET_COL]).copy()
    X = d[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = d[TARGET_COL].astype(float)
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index




def safe_monthly_rank_pct(df, value_col):
    out = pd.Series(index=df.index, dtype=float)
    for _, idx in df.groupby("rebalance_date").groups.items():
        s = pd.to_numeric(df.loc[idx, value_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = s.dropna()
        if len(valid) == 0:
            continue
        out.loc[valid.index] = valid.rank(method="average") / float(len(valid))
    return out

def add_monthly_rank_score(df, cols, direction_map, score_col):
    out = pd.Series(index=df.index, dtype=float)
    use_cols = [c for c in cols if c in df.columns]
    if len(use_cols) == 0:
        return out
    parts = []
    for col in use_cols:
        direction = direction_map.get(col, 1.0)
        r = safe_monthly_rank_pct(df, col)
        parts.append((r - 0.5) * 2.0 * direction)
    out.loc[df.index] = pd.concat(parts, axis=1).mean(axis=1)
    return out


def topn_alpha_by_score(df, score_col, n):
    rows = []
    for dt, g in df.groupby("rebalance_date"):
        d = g[[score_col, TARGET_COL]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(d) == 0:
            continue
        top = d.sort_values(score_col, ascending=False).head(min(n, len(d)))
        rows.append({"rebalance_date": dt, "topn": n, "mean_alpha": top[TARGET_COL].mean(), "count": len(top)})
    return pd.DataFrame(rows)


In [ ]:
def load_df(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.normalize()
    if TARGET_COL not in df.columns:
        raise ValueError("missing target: " + TARGET_COL)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", TARGET_COL]).copy()
    return df


df_all = load_df(DATA_PATH)
train_df = df_all[(df_all["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (df_all["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
diag_fit_df, diag_valid_df = split_diag_valid(train_df)

available_factors = [c for c in ALL_FACTOR_COLS if c in train_df.columns]
missing_factors = [c for c in ALL_FACTOR_COLS if c not in train_df.columns]

print("all:", df_all.shape, df_all["rebalance_date"].min(), df_all["rebalance_date"].max())
print("train:", train_df.shape, train_df["rebalance_date"].min(), train_df["rebalance_date"].max(), "months", train_df["rebalance_date"].nunique())
print("diag_fit:", diag_fit_df.shape, "diag_valid:", diag_valid_df.shape)
print("available factors:", len(available_factors))
print(available_factors)
print("missing factors:", missing_factors)
print(train_df[[TARGET_COL]].describe())


In [ ]:
# Single-factor IC and top-N alpha audit.
monthly_ic_rows = []
summary_rows = []
portfolio_rows = []

for factor in available_factors:
    month_ics = []
    for dt, g in train_df.groupby("rebalance_date"):
        ic = safe_rank_ic(g[factor], g[TARGET_COL])
        month_ics.append({"rebalance_date": dt, "factor": factor, "rank_ic": ic})
        monthly_ic_rows.append({"rebalance_date": dt, "factor": factor, "rank_ic": ic})
    stats = safe_ic_stats([x["rank_ic"] for x in month_ics])
    coverage = float(train_df[factor].notnull().mean())
    direction = 1.0 if (pd.isnull(stats["mean"]) or stats["mean"] >= 0) else -1.0

    tmp = train_df[["rebalance_date", TARGET_COL, factor]].copy()
    tmp["factor_score"] = safe_monthly_rank_pct(tmp, factor) * direction
    for n in TOP_LIST:
        top_df = topn_alpha_by_score(tmp, "factor_score", n)
        pstats = safe_ic_stats(top_df["mean_alpha"] if not top_df.empty else [])
        portfolio_rows.append({
            "factor": factor, "topn": n, "direction": direction,
            "mean_monthly_alpha": pstats["mean"], "monthly_alpha_ir": pstats["ir"],
            "win_rate": pstats["hit_rate"], "months": pstats["months"],
        })

    group_hits = [g for g in FACTOR_GROUPS if factor in FACTOR_GROUPS[g]]
    summary_rows.append({
        "factor": factor,
        "groups": ",".join(group_hits),
        "coverage": coverage,
        "ic_mean": stats["mean"],
        "ic_std": stats["std"],
        "ic_ir": stats["ir"],
        "ic_hit_rate": stats["hit_rate"],
        "months": stats["months"],
        "direction": direction,
    })

single_factor_ic_df = pd.DataFrame(summary_rows).sort_values(["ic_mean"], ascending=False)
monthly_factor_ic_df = pd.DataFrame(monthly_ic_rows)
single_factor_portfolio_df = pd.DataFrame(portfolio_rows).sort_values(["topn", "mean_monthly_alpha"], ascending=[True, False])

single_factor_ic_df.to_csv(os.path.join(OUT_DIR, "v48_single_factor_ic.csv"), index=False)
monthly_factor_ic_df.to_csv(os.path.join(OUT_DIR, "v48_monthly_factor_ic.csv"), index=False)
single_factor_portfolio_df.to_csv(os.path.join(OUT_DIR, "v48_single_factor_topn_alpha.csv"), index=False)

print("single factor IC top:")
print(single_factor_ic_df.head(20))
print("single factor top10 alpha top:")
print(single_factor_portfolio_df[single_factor_portfolio_df["topn"] == 10].head(20))


In [ ]:
# Yearly IC stability.
yearly_rows = []
if not monthly_factor_ic_df.empty:
    tmp = monthly_factor_ic_df.copy()
    tmp["year"] = pd.to_datetime(tmp["rebalance_date"]).dt.year
    for (factor, year), g in tmp.groupby(["factor", "year"]):
        stats = safe_ic_stats(g["rank_ic"])
        yearly_rows.append({
            "factor": factor, "year": int(year), "ic_mean": stats["mean"],
            "ic_ir": stats["ir"], "hit_rate": stats["hit_rate"], "months": stats["months"],
        })

yearly_factor_ic_df = pd.DataFrame(yearly_rows)
yearly_factor_ic_df.to_csv(os.path.join(OUT_DIR, "v48_yearly_factor_ic.csv"), index=False)
print(yearly_factor_ic_df.head(30))


In [ ]:
# Group rank-composite audit.
direction_map = dict(zip(single_factor_ic_df["factor"], single_factor_ic_df["direction"]))
group_rows = []
group_port_rows = []
group_score_df = train_df[["stock", "rebalance_date", TARGET_COL]].copy()

for group_name in sorted(FACTOR_GROUPS):
    cols = [c for c in FACTOR_GROUPS[group_name] if c in train_df.columns]
    score_col = "score_" + group_name
    group_score_df[score_col] = add_monthly_rank_score(train_df, cols, direction_map, score_col)

    month_ics = []
    for dt, g in group_score_df.groupby("rebalance_date"):
        ic = safe_rank_ic(g[score_col], g[TARGET_COL])
        month_ics.append(ic)
    stats = safe_ic_stats(month_ics)
    group_rows.append({
        "group": group_name, "factor_count": len(cols), "factors": ",".join(cols),
        "ic_mean": stats["mean"], "ic_ir": stats["ir"], "hit_rate": stats["hit_rate"], "months": stats["months"],
    })
    for n in TOP_LIST:
        top_df = topn_alpha_by_score(group_score_df, score_col, n)
        pstats = safe_ic_stats(top_df["mean_alpha"] if not top_df.empty else [])
        group_port_rows.append({
            "group": group_name, "topn": n, "mean_monthly_alpha": pstats["mean"],
            "monthly_alpha_ir": pstats["ir"], "win_rate": pstats["hit_rate"], "months": pstats["months"],
        })

group_composite_ic_df = pd.DataFrame(group_rows).sort_values("ic_mean", ascending=False)
group_composite_portfolio_df = pd.DataFrame(group_port_rows).sort_values(["topn", "mean_monthly_alpha"], ascending=[True, False])
group_composite_ic_df.to_csv(os.path.join(OUT_DIR, "v48_group_composite_ic.csv"), index=False)
group_composite_portfolio_df.to_csv(os.path.join(OUT_DIR, "v48_group_composite_topn_alpha.csv"), index=False)

print("group composite IC:")
print(group_composite_ic_df)
print("group composite top10 alpha:")
print(group_composite_portfolio_df[group_composite_portfolio_df["topn"] == 10])


In [ ]:
# LGB group-only and drop-group diagnostics on diag_valid.
def fit_eval_lgb(train_part, valid_part, cols, model_name):
    cols = [c for c in cols if c in train_part.columns]
    if len(cols) == 0:
        return {"model_name": model_name, "feature_count": 0, "diag_rank_ic": np.nan, "train_rank_ic": np.nan, "note": "no_features"}
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_part, cols)
    X_valid, y_valid, _, _ = prepare_xy(valid_part, cols, fill_values)
    if len(X_train) == 0 or len(X_valid) == 0:
        return {"model_name": model_name, "feature_count": len(cols), "diag_rank_ic": np.nan, "train_rank_ic": np.nan, "note": "empty_xy"}

    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=MODEL_FIXED_ITER,
    )
    train_pred = np.asarray(model.predict(X_train[cols], num_iteration=MODEL_FIXED_ITER)).reshape(-1)
    valid_pred = np.asarray(model.predict(X_valid[cols], num_iteration=MODEL_FIXED_ITER)).reshape(-1)
    return {
        "model_name": model_name,
        "feature_count": len(cols),
        "train_rank_ic": safe_rank_ic(y_train, train_pred),
        "diag_rank_ic": safe_rank_ic(y_valid, valid_pred),
        "note": "ok",
    }

model_rows = []
all_cols = [c for c in available_factors if c in train_df.columns]
model_rows.append(fit_eval_lgb(diag_fit_df, diag_valid_df, all_cols, "all_factors"))

for group_name in sorted(FACTOR_GROUPS):
    group_cols = [c for c in FACTOR_GROUPS[group_name] if c in train_df.columns]
    model_rows.append(fit_eval_lgb(diag_fit_df, diag_valid_df, group_cols, "only__" + group_name))
    drop_cols = [c for c in all_cols if c not in group_cols]
    model_rows.append(fit_eval_lgb(diag_fit_df, diag_valid_df, drop_cols, "drop__" + group_name))
    gc.collect()

lgb_group_diagnostic_df = pd.DataFrame(model_rows)
base_diag = lgb_group_diagnostic_df[lgb_group_diagnostic_df["model_name"] == "all_factors"]["diag_rank_ic"]
base_diag_value = float(base_diag.iloc[0]) if len(base_diag) else np.nan
lgb_group_diagnostic_df["delta_vs_all_diag_rank_ic"] = lgb_group_diagnostic_df["diag_rank_ic"] - base_diag_value
lgb_group_diagnostic_df.to_csv(os.path.join(OUT_DIR, "v48_lgb_group_diagnostic.csv"), index=False)
print(lgb_group_diagnostic_df.sort_values("diag_rank_ic", ascending=False))


In [ ]:
# Correlation cluster snapshot for available factors.
corr = train_df[available_factors].replace([np.inf, -np.inf], np.nan).corr()
corr_abs = corr.abs()
corr_rows = []
for i in range(len(available_factors)):
    for j in range(i + 1, len(available_factors)):
        a = available_factors[i]
        b = available_factors[j]
        v = corr.loc[a, b]
        if not pd.isnull(v):
            corr_rows.append({"factor_a": a, "factor_b": b, "corr": float(v), "abs_corr": float(abs(v))})

factor_corr_pairs_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False)
factor_corr_pairs_df.to_csv(os.path.join(OUT_DIR, "v48_factor_corr_pairs.csv"), index=False)
print(factor_corr_pairs_df.head(30))


In [ ]:
# Compact conclusion table for quick review.
summary_path = os.path.join(OUT_DIR, "v48_factor_audit_summary.xlsx")
try:
    with pd.ExcelWriter(summary_path) as writer:
        single_factor_ic_df.to_excel(writer, "single_factor_ic", index=False)
        single_factor_portfolio_df.to_excel(writer, "single_factor_topn", index=False)
        group_composite_ic_df.to_excel(writer, "group_composite_ic", index=False)
        group_composite_portfolio_df.to_excel(writer, "group_composite_topn", index=False)
        lgb_group_diagnostic_df.to_excel(writer, "lgb_group_diagnostic", index=False)
        factor_corr_pairs_df.head(200).to_excel(writer, "top_corr_pairs", index=False)
    print("saved xlsx:", summary_path)
except Exception as err:
    print("xlsx export skipped:", err)

print("saved csv files in:", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    print("  ", fn)


## 结论填写区

回填重点：

- 单因子最稳定的是哪些？
- top10/top30 组合最有效的是哪些？
- group-only 是否有某组能单独接近 all-factors？
- drop-group 哪组删除后下降最大？
- 是否存在高相关冗余因子？
- 下一步应该新增/删除/改造哪一类因子？
